# CTI Realm Agent Architecture Analysis (Domain-Specific)## PurposeThis notebook contains **CTI Realm domain-specific** experiments comparing different **agent architectures** (React, GH Copilot, Claude Code, etc.) across CTI Realm tasks.It focuses on analyses that are unique to CTI Realm and complement the domain-agnostic agent architecture analysis notebook (`agent_architecture_analysis.ipynb`).### Experiments| # | Experiment | What it reveals about agents ||---|-----------|------------------------------|| 1 | Checkpoint Score Heatmap by Agent | Which checkpoints each architecture excels at, per platform || 2 | CTI & MITRE Tool Usage by Agent | How agent architecture affects investigation workflow || 3 | KQL Query Analysis by Agent | Whether agents write different amounts/quality of KQL queries || 4 | Checkpoint Correlation by Agent | Do checkpoint interdependencies change with architecture? |### CTI Realm ContextCTI Realm is a **cyber threat intelligence** domain where agents:- Consult CTI reports and MITRE ATT&CK framework- Write and refine KQL (Kusto Query Language) detection queries- Develop Sigma rules for threat detection- Navigate 3 platforms: **Linux** (15 tasks), **AKS** (6 tasks), **Cloud** (4 tasks)The 5 checkpoints evaluate distinct skills:- **C0** CTI Alignment (max 1.25) — reading/using threat intelligence reports- **C1** MITRE Techniques (max 0.75) — mapping to ATT&CK framework- **C2** Data Exploration (max 1.0) — exploring data schemas and tables- **C3** Query Iteration (max 0.5) — iteratively refining queries- **C4** Detection Quality (max 6.5) — final detection rule quality (65% of total)

## Prerequisites1. Run evaluations for each agent architecture to generate `.eval` files:   ```bash   uv run inspect eval domains/cti_realm --model anthropic/claude-sonnet-4-6 -T agent=react -T dataset=cti_realm_25   uv run inspect eval domains/cti_realm --model anthropic/claude-sonnet-4-6 -T agent=copilot -T dataset=cti_realm_25   uv run inspect eval domains/cti_realm --model anthropic/claude-sonnet-4-6 -T agent=claude_code -T dataset=cti_realm_25   ```2. Place eval files in `eval_samples/cti_realm/` or `latest_experiments/cti_realm/`3. Run the **domain-agnostic** [`agent_architecture_analysis.ipynb`](agent_architecture_analysis.ipynb) first for overall scores, cost, and trajectory analysis

<a id="configuration"></a>## ConfigurationEdit the cell below to configure agents, models, and eval file paths for CTI Realm.### How to add agents or models1. Add entries to `AGENT_EVAL_LOGS` under the appropriate agent architecture2. Ensure the `.eval` file exists in `eval_samples/cti_realm/` or `latest_experiments/cti_realm/`3. Re-run the notebook

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Agent Architecture Analysis (CTI Realm-Specific)
# ══════════════════════════════════════════════════════════════════════════════
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from inspect_ai.log import read_eval_log

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from saber_analysis.data_loader import load_eval_logs, load_trajectory_data, ensure_eval_files
from saber_analysis.cost import extract_cost_rows
from saber_analysis.plots import setup_plotting, classify_score_type

setup_plotting()

# ── Domain ──
DOMAIN_NAME = 'CTI Realm'
DOMAIN_SLUG = 'cti_realm'
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
LOG_DIR = str(REPO_ROOT / 'eval_samples' / DOMAIN_SLUG)
ARTIFACTS_DIR = NOTEBOOK_DIR / 'artifacts' / 'cti_realm_agent'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# AGENT REGISTRY — {agent: {display_name: filename.eval}}
# ══════════════════════════════════════════════════════════════════════════════
AGENT_EVAL_LOGS = {
    'React': {
        'Claude Sonnet 4.6': 'sonnet_4.6_cti_realm_25.eval',
    },
    'GH Copilot': {
        'Claude Sonnet 4.6': 'sonnet_4.6_copilot.eval',
    },
    'Claude Code': {
        'Claude Sonnet 4.6': 'sonnet_4.6_claude_code.eval',
    },
}

AGENT_COLORS = {
    'React':       '#2CA02C',
    'GH Copilot':  '#E74C3C',
    'Claude Code': '#3498DB',
}

MODEL_COLORS = {
    'Claude Haiku 4.5':  '#17BECF',
    'Claude Sonnet 4.6': '#F39C12',
    'Claude Opus 4.6':   '#E74C3C',
    'GPT-5.4':           '#2CA02C',
    'GPT-5.4-mini':      '#7B4FBF',
}

# Flatten for data loading
EVAL_LOGS = {}
DISPLAY_ORDER = []
COLORS = {}
AGENT_OF = {}
MODEL_OF = {}
for agent, models in AGENT_EVAL_LOGS.items():
    for model, filename in models.items():
        display = f'{model} ({agent})'
        EVAL_LOGS[display] = filename
        DISPLAY_ORDER.append(display)
        # Color by agent when single model, by model when multi-model
        n_unique_models = len(set(m for a in AGENT_EVAL_LOGS.values() for m in a))
        COLORS[display] = AGENT_COLORS.get(agent, '#888888') if n_unique_models == 1 else MODEL_COLORS.get(model, '#888888')
        AGENT_OF[display] = agent
        MODEL_OF[display] = model

MODEL_ORDER = DISPLAY_ORDER
N_SAMPLES = 25  # cti_realm_25 dataset

# CTI Realm groups by platform
GROUP_FN = lambda sid: sid.split('_')[0]
GROUP_LABEL = 'Platform'

# CTI Realm checkpoint definitions
CHECKPOINT_NAMES = {
    'c0_cti_analysis': 'C0: CTI Alignment',
    'c1_mitre_techniques': 'C1: MITRE Techniques',
    'c2_data_exploration': 'C2: Data Exploration',
    'c3_query_iteration': 'C3: Query Iteration',
    'c4_detection_quality': 'C4: Detection Quality',
}
CHECKPOINT_MAX_SCORES = {
    'c0_cti_analysis': 1.25,
    'c1_mitre_techniques': 0.75,
    'c2_data_exploration': 1.0,
    'c3_query_iteration': 0.5,
    'c4_detection_quality': 6.5,
}

# CTI Realm domain-specific tools
CTI_TOOLS = [
    'list_cti_report_tags', 'get_cti_reports_by_tag',
    'search_mitre_techniques', 'search_sigma_rules',
    'execute_kql_query', 'list_kusto_tables', 'get_table_schema', 'sample_table_data',
    'validate_output_json',
]

ensure_eval_files(
    EVAL_LOGS, LOG_DIR, DOMAIN_SLUG,
    fallback_dirs=[
        str(REPO_ROOT / 'latest_experiments' / DOMAIN_SLUG),
        str(REPO_ROOT / 'latest_experiments' / DOMAIN_SLUG / 'agent_arch'),
    ],
)

print(f'Domain: {DOMAIN_NAME}')
print(f'  Agent configs: {len(EVAL_LOGS)} ({", ".join(DISPLAY_ORDER)})')
print(f'  Log dir: {LOG_DIR}')

## Data Loading### What this doesParses all `.eval` ZIP archives from the agent x model matrix and extracts:- **`samples_df`**: per-sample `saber_overall` scores with config and platform group columns- **`subtasks_df`**: per-subtask scores (C0–C4, aggregate) for each sample- **`overall_df`**: per-config mean and standard error- **`traj_df`**: trajectory data with tool call counts and timing

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
samples_df, subtasks_df, overall_df, model_overall = load_eval_logs(
    EVAL_LOGS, LOG_DIR, MODEL_ORDER, group_fn=GROUP_FN
)

print(f'\nLoaded {len(samples_df)} sample scores across {samples_df["model"].nunique()} agent configs')
print(f'Loaded {len(subtasks_df)} sub-task scores')
if GROUP_FN:
    print(f'Platforms: {sorted(samples_df["group"].unique())}')
print()
print('Overall Scores per Agent Config')
for m in MODEL_ORDER:
    if m in model_overall:
        r = overall_df.loc[m]
        print(f'  {m:35s}  mean={r["mean"]:.4f}  stderr={r["stderr"]:.4f}')

subtasks_df['category'] = subtasks_df['score_type'].apply(classify_score_type)

# Load trajectory data
from collections import Counter
traj_df = load_trajectory_data(EVAL_LOGS, LOG_DIR, MODEL_ORDER, group_fn=GROUP_FN)
traj_df['tool_counts'] = traj_df['tool_counts'].apply(
    lambda d: dict(Counter({k.lower(): v for k, v in d.items()}))
)
all_tools = sorted({t for tc in traj_df['tool_counts'] for t in tc})
for tool in all_tools:
    traj_df[f'tool_{tool}'] = traj_df['tool_counts'].apply(lambda d, t=tool: d.get(t, 0))

active_traj = [m for m in MODEL_ORDER if m in traj_df['model'].unique()]

print(f'\nLoaded trajectory data: {len(traj_df)} samples x {traj_df["model"].nunique()} configs')
print(f'  Tools observed: {all_tools}')

---## 1. Checkpoint Score Heatmap by Agent Architecture### What this measuresA heatmap showing **mean checkpoint scores per platform per agent configuration**. Each checkpoint (C0–C4) captures a distinct CTI skill, and different agent architectures may strengthen or weaken specific capabilities.### How to read the chart- **Left panel**: Raw checkpoint scores per agent config across platforms — brighter green = higher score- **Right panel**: Agent architecture difference from React baseline — green = agent outperforms React, red = underperforms- Rows = agent configurations; columns = platform x checkpoint combinations### Key questions- Does agent architecture affect which checkpoints succeed? (e.g., does Copilot improve C4 detection writing but hurt C0 CTI alignment?)- Are platform-specific advantages consistent across architectures?- Which checkpoint shows the largest architecture sensitivity?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1: Checkpoint Score Heatmap by Agent Architecture
# ══════════════════════════════════════════════════════════════════════════════
checkpoints = list(CHECKPOINT_NAMES.keys())
active_configs = [m for m in MODEL_ORDER if m in model_overall]

# Build checkpoint × platform × config matrix
ckpt_platform = []
for m in active_configs:
    m_sub = subtasks_df[subtasks_df['model'] == m]
    m_samp = samples_df[samples_df['model'] == m]
    for platform in sorted(samples_df['group'].unique()):
        platform_sids = m_samp[m_samp['group'] == platform]['sample_id'].unique()
        for ck in checkpoints:
            ck_scores = m_sub[(m_sub['score_type'] == ck) & (m_sub['sample_id'].isin(platform_sids))]['score']
            if len(ck_scores) > 0:
                # Normalize by max score
                max_score = CHECKPOINT_MAX_SCORES.get(ck, 1.0)
                ckpt_platform.append({
                    'model': m, 'platform': platform, 'checkpoint': ck,
                    'score': ck_scores.mean(), 'normalized': ck_scores.mean() / max_score,
                })

ckpt_df = pd.DataFrame(ckpt_platform)

# Create combined column label: "Platform - Checkpoint"
ckpt_df['col_label'] = ckpt_df['platform'] + ' - ' + ckpt_df['checkpoint'].map(
    lambda ck: CHECKPOINT_NAMES[ck].split(': ')[1][:15])

# Pivot for heatmap
heat_pivot = ckpt_df.pivot_table(index='model', columns='col_label', values='normalized', aggfunc='mean')
heat_pivot = heat_pivot.reindex(active_configs)

# Sort columns by platform then checkpoint
col_order = []
for platform in sorted(samples_df['group'].unique()):
    for ck in checkpoints:
        short = CHECKPOINT_NAMES[ck].split(': ')[1][:15]
        label = f'{platform} - {short}'
        if label in heat_pivot.columns:
            col_order.append(label)
heat_pivot = heat_pivot[col_order]

fig, ax = plt.subplots(figsize=(max(16, len(col_order) * 0.9), max(5, len(active_configs) * 1.2)))
im = ax.imshow(heat_pivot.values, cmap='YlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(col_order)))
ax.set_xticklabels(col_order, fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(len(active_configs)))
ax.set_yticklabels(active_configs, fontsize=9)

for i, m in enumerate(active_configs):
    for j, col in enumerate(col_order):
        val = heat_pivot.loc[m, col]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    fontweight='bold', color='white' if val > 0.6 else 'black')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Normalized Checkpoint Score (0–1)', fontsize=10)
ax.set_title(f'Checkpoint Scores by Platform x Agent Architecture — {DOMAIN_NAME}', fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'checkpoint_heatmap_by_agent.png', bbox_inches='tight')
plt.show()

# Summary: which checkpoints are most architecture-sensitive?
print('\nCheckpoint Architecture Sensitivity (std across configs):')
for ck in checkpoints:
    ck_short = CHECKPOINT_NAMES[ck]
    ck_cols = [c for c in col_order if ck_short.split(': ')[1][:15] in c]
    if ck_cols:
        vals = heat_pivot[ck_cols].mean(axis=1)
        print(f'  {ck_short:30s}  '
              f'min={vals.min():.3f}  max={vals.max():.3f}  '
              f'spread={vals.max()-vals.min():.3f}  '
              f'({active_configs[vals.argmin()]} → {active_configs[vals.argmax()]})')

---## 2. CTI & MITRE Tool Usage by Agent Architecture### What this measuresExamines how each agent architecture leverages the **9 CTI-specific tools**:- **(a) Mean Tool Calls per Sample** — grouped bar chart of all CTI tools per agent config- **(b) Investigation Workflow Funnel** — traces the expected flow: CTI reports → MITRE mapping → KQL querying → validation- **(c) KQL Query Count Distribution** — violin plot per agent config- **(d) KQL Queries vs. C4 Detection Quality** — scatter testing whether more queries → better detection### Key questions- Do agent architectures produce different investigation strategies?- Does one architecture encourage more systematic use of CTI reports before querying?- Which architecture produces the most efficient KQL querying (fewer queries for same C4 score)?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2: CTI & MITRE Tool Usage by Agent Architecture
# ══════════════════════════════════════════════════════════════════════════════
# Build per-sample tool counts for CTI tools
cti_records = []
for m in MODEL_ORDER:
    mdf = traj_df[traj_df['model'] == m]
    for _, row in mdf.iterrows():
        rec = {'model': m, 'sample_id': row['sample_id']}
        for t in CTI_TOOLS:
            col = f'tool_{t}'
            rec[t] = int(row[col]) if col in row.index else 0
        cti_records.append(rec)

cti_df = pd.DataFrame(cti_records)

# Get C4 scores for scatter plot
c4_lookup = subtasks_df[subtasks_df['score_type'] == 'c4_detection_quality'][['model', 'sample_id', 'score']].copy()
c4_lookup = c4_lookup.rename(columns={'score': 'c4_score'})

fig, axes = plt.subplots(2, 2, figsize=(20, 14))

# (a) Mean calls per tool per agent config — grouped bar
ax = axes[0, 0]
x = np.arange(len(CTI_TOOLS))
n_configs = len(active_traj)
width = min(0.2, 0.85 / n_configs)
for i, m in enumerate(active_traj):
    mdf = cti_df[cti_df['model'] == m]
    means = [mdf[t].mean() for t in CTI_TOOLS]
    ax.bar(x + i * width, means, width, label=m, color=COLORS[m], edgecolor='black', linewidth=0.3)
ax.set_xticks(x + width * (n_configs - 1) / 2)
ax.set_xticklabels([t.replace('_', '\n') for t in CTI_TOOLS], fontsize=7, rotation=30, ha='right')
ax.set_ylabel('Mean Calls per Sample')
ax.set_title('(a) Mean CTI Tool Calls per Sample', fontweight='bold')
ax.legend(fontsize=7)

# (b) Investigation workflow funnel
ax = axes[0, 1]
funnel_tools = ['list_cti_report_tags', 'get_cti_reports_by_tag', 'search_mitre_techniques',
                'search_sigma_rules', 'execute_kql_query', 'validate_output_json']
for m in active_traj:
    mdf = cti_df[cti_df['model'] == m]
    means = [mdf[t].mean() for t in funnel_tools]
    ax.plot(range(len(funnel_tools)), means, 'o-', label=m, color=COLORS[m], linewidth=2, markersize=8)
ax.set_xticks(range(len(funnel_tools)))
ax.set_xticklabels([t.replace('_', '\n') for t in funnel_tools], fontsize=7)
ax.set_ylabel('Mean Calls per Sample')
ax.set_title('(b) Investigation Workflow Funnel', fontweight='bold')
ax.legend(fontsize=7)

# (c) KQL query count distribution
ax = axes[1, 0]
for i, m in enumerate(active_traj):
    data = cti_df[cti_df['model'] == m]['execute_kql_query'].values
    if len(data) > 0:
        vp = ax.violinplot([data], positions=[i], showmeans=True, showmedians=True, widths=0.7)
        for pc in vp['bodies']:
            pc.set_facecolor(COLORS[m]); pc.set_alpha(0.7)
        for part in ('cmaxes', 'cmins', 'cbars', 'cmeans', 'cmedians'):
            if part in vp:
                vp[part].set_edgecolor('black'); vp[part].set_linewidth(0.8)
        ax.text(i, data.max() + 0.3, f'μ={np.mean(data):.1f}', ha='center', fontsize=8, fontweight='bold')
ax.set_xticks(range(len(active_traj)))
ax.set_xticklabels(active_traj, fontsize=8, rotation=20, ha='right')
ax.set_ylabel('KQL Queries per Sample')
ax.set_title('(c) KQL Query Count Distribution', fontweight='bold')

# (d) KQL count vs C4 score
ax = axes[1, 1]
cti_c4 = cti_df.merge(c4_lookup, on=['model', 'sample_id'], how='left')
for m in active_traj:
    mdf = cti_c4[cti_c4['model'] == m]
    ax.scatter(mdf['execute_kql_query'], mdf['c4_score'], label=m,
               color=COLORS[m], alpha=0.6, s=80, edgecolors='w', linewidths=0.5)
ax.set_xlabel('KQL Query Count per Sample')
ax.set_ylabel('C4 Detection Quality Score')
ax.set_title('(d) KQL Queries vs C4 Detection Quality', fontweight='bold')
ax.legend(fontsize=8)

fig.suptitle(f'CTI Tool Usage by Agent Architecture — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'cti_tool_usage_by_agent.png', bbox_inches='tight')
plt.show()

# Summary table
print('\nCTI Tool Usage Summary per Agent Config')
print(f'{"Config":35s} {"KQL":>5s} {"CTI Rpt":>8s} {"MITRE":>7s} {"Sigma":>7s} {"Validate":>9s}')
print('-' * 75)
for m in active_traj:
    mdf = cti_df[cti_df['model'] == m]
    print(f'{m:35s} {mdf["execute_kql_query"].mean():5.1f} '
          f'{mdf["get_cti_reports_by_tag"].mean():8.1f} '
          f'{mdf["search_mitre_techniques"].mean():7.1f} '
          f'{mdf["search_sigma_rules"].mean():7.1f} '
          f'{mdf["validate_output_json"].mean():9.1f}')

---## 3. KQL Query Quality Analysis by Agent Architecture### What this measuresDeeper analysis of KQL querying behavior per agent configuration:- **(a) Total KQL Queries** — absolute volume per agent config- **(b) Per-Sample Distribution** — box plot showing query count variance- **(c) KQL Efficiency** — C4 score per KQL query (higher = more efficient querying)### Key questions- Which architecture writes the most KQL queries? Is more always better?- Which architecture is the most KQL-efficient (highest C4 score per query)?- Do architectures with fewer but more targeted queries outperform those that brute-force?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3: KQL Query Quality Analysis by Agent Architecture
# ══════════════════════════════════════════════════════════════════════════════
kql_records = []
for m in active_traj:
    mdf = traj_df[traj_df['model'] == m]
    for _, row in mdf.iterrows():
        n_kql = int(row.get('tool_execute_kql_query', 0))
        kql_records.append({
            'model': m, 'sample_id': row['sample_id'],
            'n_kql_total': n_kql,
        })

kql_df = pd.DataFrame(kql_records)
kql_df = kql_df.merge(c4_lookup, on=['model', 'sample_id'], how='left')

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# (a) Total KQL queries per config
ax = axes[0]
x = np.arange(len(active_traj))
totals = [kql_df[kql_df['model'] == m]['n_kql_total'].sum() for m in active_traj]
for i, (m, t) in enumerate(zip(active_traj, totals)):
    ax.bar(i, t, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.text(i, t + max(totals)*0.02, str(t), ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(active_traj, fontsize=9, rotation=20, ha='right')
ax.set_ylabel('Total KQL Queries')
ax.set_title('(a) Total KQL Queries per Config', fontweight='bold')

# (b) Per-sample box plot
ax = axes[1]
bp_data = [kql_df[kql_df['model'] == m]['n_kql_total'].values for m in active_traj]
bp = ax.boxplot(bp_data, labels=[m.replace('Claude ', '').replace('GPT-', 'GPT') for m in active_traj],
                patch_artist=True)
for patch, m in zip(bp['boxes'], active_traj):
    patch.set_facecolor(COLORS[m]); patch.set_alpha(0.7)
ax.set_ylabel('KQL Queries per Sample')
ax.set_title('(b) Per-Sample KQL Query Count', fontweight='bold')
ax.set_xticklabels([m.replace('Claude ', '').replace('GPT-', 'GPT') for m in active_traj],
                    fontsize=9, rotation=20, ha='right')

# (c) KQL efficiency: C4 score per KQL query
ax = axes[2]
efficiencies = []
for m in active_traj:
    mdf = kql_df[kql_df['model'] == m]
    mean_kql = mdf['n_kql_total'].mean()
    mean_c4 = mdf['c4_score'].mean() if 'c4_score' in mdf.columns else 0
    eff = mean_c4 / mean_kql if mean_kql > 0 else 0
    efficiencies.append(eff)
for i, (m, eff) in enumerate(zip(active_traj, efficiencies)):
    ax.bar(i, eff, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.text(i, eff + max(efficiencies)*0.02, f'{eff:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_xticks(range(len(active_traj)))
ax.set_xticklabels([m.replace('Claude ', '').replace('GPT-', 'GPT') for m in active_traj],
                    fontsize=9, rotation=20, ha='right')
ax.set_ylabel('C4 Score / KQL Query')
ax.set_title('(c) KQL Efficiency (C4 per Query)', fontweight='bold')
ax.set_ylim(0, max(efficiencies) * 1.3 if efficiencies else 1)

fig.suptitle(f'KQL Query Analysis by Agent Architecture — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'kql_analysis_by_agent.png', bbox_inches='tight')
plt.show()

# Summary
print('\nKQL Efficiency Summary:')
print(f'{"Config":35s} {"Total KQL":>10s} {"Mean/Sample":>12s} {"Mean C4":>8s} {"C4/Query":>10s}')
print('-' * 80)
for m, eff in zip(active_traj, efficiencies):
    mdf = kql_df[kql_df['model'] == m]
    mean_kql = mdf['n_kql_total'].mean()
    mean_c4 = mdf['c4_score'].mean() if 'c4_score' in mdf.columns else 0
    print(f'{m:35s} {mdf["n_kql_total"].sum():10d} {mean_kql:12.1f} {mean_c4:8.2f} {eff:10.3f}')

---## 4. Checkpoint Correlation by Agent Architecture### What this measuresCompares **checkpoint interdependencies** across agent architectures using Spearman rank correlation. Key questions:- Do checkpoint relationships change with agent architecture?- Does one architecture produce more aligned checkpoint performance (higher inter-checkpoint correlation)?- Is the C2 ↔ C4 relationship (data exploration → detection quality) stronger for some architectures?### How to read the chart- One correlation matrix per agent config — compare patterns across panels- Red = positive correlation, blue = negative- Consistent patterns across agents = intrinsic task property; different patterns = architecture effect

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 4: Checkpoint Correlation by Agent Architecture
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr

checkpoints = list(CHECKPOINT_NAMES.keys())
active_configs = [m for m in MODEL_ORDER if m in model_overall]
n_configs = len(active_configs)

fig, axes = plt.subplots(1, n_configs, figsize=(7 * n_configs, 6), squeeze=False)

for idx, m in enumerate(active_configs):
    ax = axes[0, idx]

    # Build checkpoint score matrix for this config
    m_sub = subtasks_df[subtasks_df['model'] == m]
    ckpt_records = []
    for sid in m_sub['sample_id'].unique():
        row = {'sample_id': sid}
        sid_sub = m_sub[m_sub['sample_id'] == sid]
        for ck in checkpoints:
            ck_scores = sid_sub[sid_sub['score_type'] == ck]['score']
            row[ck] = ck_scores.values[0] if len(ck_scores) > 0 else np.nan
        ckpt_records.append(row)
    ckpt_config_df = pd.DataFrame(ckpt_records)

    # Spearman correlation
    corr = ckpt_config_df[checkpoints].corr(method='spearman')

    labels = [CHECKPOINT_NAMES[ck].split(': ')[1][:12] for ck in checkpoints]
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(checkpoints)))
    ax.set_xticklabels(labels, fontsize=8, rotation=30, ha='right')
    ax.set_yticks(range(len(checkpoints)))
    ax.set_yticklabels(labels, fontsize=8)

    for i in range(len(checkpoints)):
        for j in range(len(checkpoints)):
            val = corr.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                        fontweight='bold' if abs(val) > 0.3 else 'normal',
                        color='white' if abs(val) > 0.6 else 'black')

    short = m.replace('Claude ', '').replace('GPT-', 'GPT')
    ax.set_title(short, fontweight='bold', fontsize=10)

# Add single colorbar
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Spearman ρ')

fig.suptitle(f'Checkpoint Correlations by Agent Architecture — {DOMAIN_NAME}',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.savefig(ARTIFACTS_DIR / 'checkpoint_correlations_by_agent.png', bbox_inches='tight')
plt.show()

# Key correlation comparison
print('\nKey Checkpoint Correlations by Agent Config:')
print(f'{"Config":35s} {"C0↔C4":>7s} {"C2↔C4":>7s} {"C1↔C4":>7s} {"C0↔C1":>7s}')
print('-' * 70)
for m in active_configs:
    m_sub = subtasks_df[subtasks_df['model'] == m]
    ckpt_records = []
    for sid in m_sub['sample_id'].unique():
        row = {}
        sid_sub = m_sub[m_sub['sample_id'] == sid]
        for ck in checkpoints:
            ck_scores = sid_sub[sid_sub['score_type'] == ck]['score']
            row[ck] = ck_scores.values[0] if len(ck_scores) > 0 else np.nan
        ckpt_records.append(row)
    df = pd.DataFrame(ckpt_records)

    pairs = [
        ('c0_cti_analysis', 'c4_detection_quality'),
        ('c2_data_exploration', 'c4_detection_quality'),
        ('c1_mitre_techniques', 'c4_detection_quality'),
        ('c0_cti_analysis', 'c1_mitre_techniques'),
    ]
    vals = []
    for c1, c2 in pairs:
        valid = df[[c1, c2]].dropna()
        if len(valid) > 2:
            rho, _ = spearmanr(valid[c1], valid[c2])
            vals.append(f'{rho:+.3f}')
        else:
            vals.append('  n/a')
    print(f'{m:35s} {"  ".join(vals)}')

---## Next Steps### Companion notebooks- Run the **domain-agnostic** [`agent_architecture_analysis.ipynb`](agent_architecture_analysis.ipynb) for overall score comparisons, cost analysis, tool usage, and trajectory analysis across agent architectures (Experiments 1–12).- See [`cti_realm_analysis.ipynb`](cti_realm_analysis.ipynb) for model-comparison analysis in the CTI Realm domain (without the agent architecture dimension).### Extending this analysis1. After more agent evaluations complete (e.g., more models × agents), re-run for richer comparisons2. Compare checkpoint heatmaps to identify which agent architectures best support specific CTI skills3. Investigate KQL query patterns to understand how architecture affects investigation strategy4. Cross-reference with the domain-agnostic notebook for cost and efficiency layering### Data loading notesAll log parsing uses **inspect_ai's typed log API**:- `read_eval_log(header_only=True)` for run-level results and stats- `read_eval_log_sample_summaries()` for per-sample scores, timing, and model usage- `read_eval_log_sample(id=...)` for full message/tool-call detail per sample